# Identify analyses to rerun

Build the full set of country/analysis combinations that should be used, drop those already usable on disk, and write the remainder to `data/config/rerun_pairs.json` for `scripts/massive/jtrauer/rerun_countries.py`.

A combination is in scope if it was requested (`get_analyses_for_country`) and was not skipped (no scaler data). Complete means `store_outputs` finished (`updates.h5` present).

Jobs are searched in the order listed below. The first job has precedence because its methods are more recent: if a combination is complete there, that copy is the one to use. Later jobs only fill gaps. Remaining work is requested combinations that are complete in none of the listed jobs.

The inventory cell reports how much of each job is actually on disk. If a folder is only a partial copy, combinations that finished remotely but are missing locally will be treated as still needed.


In [ ]:
import json
import pandas as pd

from emu_renewal.constants import ANALYSIS_TYPES, DATA_PATH, OUTPUTS_PATH
from emu_renewal.run import get_analyses_for_country

In [ ]:
job_ids = ["59746206", "59597639"]
countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))


def job_inventory(run_id):
    run_path = OUTPUTS_PATH / run_id
    present = [p.name for p in run_path.iterdir() if p.is_dir()] if run_path.exists() else []
    n_complete = sum(
        (run_path / iso3 / analysis / "updates.h5").exists()
        for iso3 in present
        for analysis in ANALYSIS_TYPES
    )
    return {"countries on disk": len(present), "complete analyses": n_complete}


pd.DataFrame({job: job_inventory(job) for job in job_ids})

In [ ]:
def classify_analysis(iso3, analysis, run_path, log_text):
    """Match run.py: complete if store_outputs finished, skipped if ScalerException."""
    if (run_path / iso3 / analysis / "updates.h5").exists():
        return "complete"
    if log_text is None:
        return "no log"
    if f"{analysis} data not available" in log_text:
        return "skipped"
    return "not run"


def classify_job(run_id):
    run_path = OUTPUTS_PATH / run_id
    status = pd.DataFrame(index=countries, columns=ANALYSIS_TYPES)
    for iso3 in countries:
        log_path = run_path / iso3 / "run.log"
        log_text = log_path.read_text() if log_path.exists() else None
        requested_types = get_analyses_for_country(iso3)
        for analysis in ANALYSIS_TYPES:
            if analysis not in requested_types:
                status.loc[iso3, analysis] = "not requested"
            else:
                status.loc[iso3, analysis] = classify_analysis(iso3, analysis, run_path, log_text)
    return status


statuses = {job: classify_job(job) for job in job_ids}
pd.concat(
    {job: s.apply(pd.Series.value_counts).fillna(0).astype(int) for job, s in statuses.items()},
    axis=1,
).fillna(0).astype(int)

In [ ]:
def pairs_from_mask(mask):
    stacked = mask.stack()
    return list(stacked[stacked].index)


requested = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
for iso3 in countries:
    requested.loc[iso3, get_analyses_for_country(iso3)] = True

skipped = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
claimed = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
summary = {"requested": int(requested.to_numpy().sum())}
for job in job_ids:
    skipped = skipped | (statuses[job] == "skipped")
    complete = statuses[job] == "complete"
    summary[f"available from {job}"] = int((requested & complete & ~claimed).to_numpy().sum())
    claimed = claimed | complete

summary["skipped"] = int((requested & skipped).to_numpy().sum())
need = requested & ~skipped & ~claimed
rerun_pairs = sorted(pairs_from_mask(need))
summary["remaining"] = len(rerun_pairs)
pd.Series(summary)

In [ ]:
pd.Series([analysis for _, analysis in rerun_pairs]).value_counts()

In [ ]:
json.dump(rerun_pairs, open(DATA_PATH / "config/rerun_pairs.json", "w"))